In [1]:
import hashlib
import json

import photolab as lab
from huggingface_hub import HfApi

REPO = "google/siglip-base-patch16-224"
info = HfApi().model_info(REPO, files_metadata=True)
print("revision:", info.sha)
print("licence:", info.card_data.get("license") if info.card_data else "NOT STATED - stop here")
for f in info.siblings:
    print(f"  {f.rfilename:28} {f.size / 1e6:8.1f} MB")

revision: 7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed
licence: apache-2.0
  .gitattributes                    0.0 MB
  README.md                         0.0 MB
  config.json                       0.0 MB
  model.safetensors               812.7 MB
  preprocessor_config.json          0.0 MB
  pytorch_model.bin               812.8 MB
  special_tokens_map.json           0.0 MB
  spiece.model                      0.8 MB
  tokenizer.json                    2.4 MB
  tokenizer_config.json             0.0 MB


In [6]:
from huggingface_hub import snapshot_download

REVISION = "7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed"  # pinned 7 Oct 2026
WANT = [
    "config.json",
    "model.safetensors",
    "preprocessor_config.json",
    "special_tokens_map.json",
    "spiece.model",
    "tokenizer_config.json",
]
snapshot_download(REPO, revision=REVISION, allow_patterns=WANT, local_dir=lab.MODEL_DIR)
print(sorted(p.name for p in lab.MODEL_DIR.iterdir()))

Fetching 6 files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 588.23it/s]

['.cache', 'config.json', 'model.safetensors', 'preprocessor_config.json', 'special_tokens_map.json', 'spiece.model', 'tokenizer_config.json']


In [7]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


hub = {f.rfilename: f for f in info.siblings}
files = {}
for name in WANT:
    path = lab.MODEL_DIR / name
    digest = sha256(path)
    # the Hub fingerprints only large files (Git LFS); small ones have no record
    expected = hub[name].lfs.sha256 if hub[name].lfs else None
    assert expected in (None, digest), f"{name}: the download does not match the Hub"
    files[name] = {"bytes": path.stat().st_size, "sha256": digest}
    print(f"  {name:28} {digest[:16]}...  {'checked against the Hub' if expected else 'recorded'}")

  config.json                  cd85b3d288297228...  recorded
  model.safetensors            2c63cb7d1f2e95ba...  checked against the Hub
  preprocessor_config.json     d11ccb80f15d358a...  recorded
  special_tokens_map.json      2b6a1ff67a27e0df...  recorded
  spiece.model                 1e5036bed065526c...  checked against the Hub
  tokenizer_config.json        d6423dae508cc3a1...  recorded


In [8]:
lock = {
    "repo": REPO,
    "revision": REVISION,
    "licence": info.card_data.get("license") if info.card_data else None,
    "files": files,
}
(lab.HERE / "MODEL_LOCK.json").write_text(json.dumps(lock, indent=1) + "\n")
print(json.dumps(lock, indent=1)[:400])

{
 "repo": "google/siglip-base-patch16-224",
 "revision": "7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed",
 "licence": "apache-2.0",
 "files": {
  "config.json": {
   "bytes": 432,
   "sha256": "cd85b3d28829722820bcb89a2cfbb4160e55fd359249a3044da724166a8d9688"
  },
  "model.safetensors": {
   "bytes": 812672320,
   "sha256": "2c63cb7d1f2e95ba501893cbb8faeb4ea9a3af295498d35097126228659c2af8"
  },
  "pre
